In [1]:
import pandas as pd
import numpy as np
import torch
import random
import torch.nn.functional as F

AA_IDS = [
    'A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y'
]

# molecular weights are in kilodaltons
MOLECULAR_WEIGHTS = {
    'A': 0.09,
    'C': 0.12,
    'D': 0.13,
    'E': 0.15,
    'F': 0.17,
    'G': 0.08,
    'H': 0.16,
    'I': 0.13,
    'K': 0.15,
    'L': 0.13,
    'M': 0.15,
    'N': 0.13,
    'P': 0.12,
    'Q': 0.15,
    'R': 0.17,
    'S': 0.11,
    'T': 0.12,
    'V': 0.12,
    'W': 0.20,
    'Y': 0.18,
}

# net charges at physiological pH ~7.4
NET_CHARGES = {
    'A': 0,
    'C': 0,
    'D': -1,
    'E': -1,
    'F': 0,
    'G': 0,
    'H': 0,
    'I': 0,
    'K': +1,
    'L': 0,
    'M': 0,
    'N': 0,
    'P': 0,
    'Q': 0,
    'R': +1,
    'S': 0,
    'T': 0,
    'V': 0,
    'W': 0,
    'Y': 0
}

# isoelectric point pHs from peptide web
# NOTE: will retrain model since I was using old textbook values with one outdated value -> shouldn't affect model accuracy much
ISOELECTRIC_PTS = {
    'A': 6.02,
    'C': 5.02,
    'D': 2.98,
    'E': 3.22,
    'F': 5.48,
    'G': 5.97,
    'H': 7.59,
    'I': 5.98,
    'K': 9.47,
    'L': 5.98,
    'M': 5.75,
    'N': 5.41,
    'P': 6.30,
    'Q': 5.65,
    'R': 10.76,
    'S': 5.68,
    'T': 5.60,
    'V': 5.97,
    'W': 5.94,
    'Y': 5.66
}

# hydrophobicity levels - pH 7
# +ve values and up = hydrophobic
# around 0 = neutral
# -ve values = hydrophilic
HYDROPHOBICITY_IDXS = {
    'A': 41,
    'C': 49,
    'D': -55,
    'E': -31,
    'F': 100,
    'G': 0,
    'H': 8,
    'I': 100,
    'K': -23,
    'L': 97,
    'M': 74,
    'N': -28,
    'P': -46,
    'Q': -10,
    'R': -14,
    'S': -5,
    'T': 13,
    'V': 76,
    'W': 97,
    'Y': 63
}


# individual half lives
HALF_LIFE = {
    'A': 4.4,
    'C': 1.2,
    'D': 1.1,
    'E': 1,
    'F': 1,
    'G': 30,
    'H': 3.5,
    'I': 20,
    'K': 1.3,
    'L': 5.5,
    'M': 30,
    'N': 1.4,
    'P': 21,
    'Q': 0.8,
    'R': 1,
    'S': 1.9,
    'T': 7.2,
    'V': 100,
    'W': 2.8,
    'Y': 2.8
}


# pseudo molecular fingerprint accounting for biochemical / functional group standout features
# check function_fp_scheme.txt for more understanding
FUNCTIONAL_FP = {
    "A": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "G": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "D": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "E": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "K": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "R": [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "H": [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "N": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "Q": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "C": [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    "M": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
    "F": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "Y": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
    "W": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "L": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "I": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "V": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "S": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "T": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "P": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
}

In [3]:
# do a bit of data exploration
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
samples = AA3D_df.iloc[:10]
max_limit = AA3D_df["3d_struct"].explode().max()
print(f"limit: {max_limit}") # let's go like 3 Angstroms above that just in case, we do want it to make new molecules


limit: [array([9.99124908, 0.70824814, 2.56974602]), array([ 9.33625031, -1.8057518 , -0.18525124]), array([ 9.74625015, -5.43974686, -4.75625038]), array([ 4.0692482 ,  2.61224747, -9.07425117]), array([ 3.91825104,  3.28425217, -5.35925484]), array([ 1.62125015,  3.64925003, -2.33725166]), array([-2.07875061,  2.96625137, -3.20825386]), array([-3.00775146,  6.03725052, -1.10925484]), array([-0.22574997,  8.14925003, -2.63424873]), array([1.26424789, 8.68925095, 0.79274559]), array([6.2472496 , 6.68425369, 1.18275261]), array([-4.55875015,  0.13924789, -8.34025002]), array([-1.37075043,  0.50525284, -6.29125404]), array([ 0.29224777,  1.18625259, -9.62024879]), array([ -0.03475189,   4.79024887, -10.94725227])]


In [14]:
class SpAAtialEnc:

    def __init__(self, resolution):
        self.resolution = resolution
        self.angstrom_lim = max_limit + 3   # maximum angstrom range found + some extra for context enrichment
        self.angstrom_inc = float(8 / resolution)

        self.hit_layer = None  # [resolution, 2] -> on and off
        self.coord_layer = None # [1, 3] -> 3D vector

    def init_spAAtial(self):
        self.make_hit_layer()
        self.make_coord_layer()

    def make_hit_layer(self):
        self.hit_layer = torch.rand(size=(self.resolution, 2))    # literally just a tensor with a hit or no hit cell

    def make_coord_layer(self):
        self.coord_layer = torch.randn(size=(1, 3))

    def radial_sequence(self):
        pass


def test_resolution():
    spatial_module = SpAAtialEnc(resolution = 1000)
    spatial_module.make_hit_layer()
    print(spatial_module.hit_layer)
test_resolution()

{'AA_IDs': ['E', 'K', 'I', 'G', 'E', 'G', 'T', 'G', 'V', 'V', 'Y', 'K', 'V', 'V', 'A', 'L', 'K', 'V', 'K', 'L', 'V', 'F', 'E', 'F', 'L', 'H', 'Q', 'D', 'L', 'K', 'K', 'K', 'P', 'Q', 'N', 'L', 'L', 'I', 'K', 'L', 'A', 'D', 'L', 'E'], 'AA_3Ds': [[-1.41055556, -5.20638964, -10.3391666], [-2.91055553, -6.43038866, -7.08816667], [-0.54055558, -4.75238917, -4.62616642], [-1.58455555, -7.00138781, -1.71916684], [-4.40555565, -7.52238962, 0.8278331], [-4.99455564, -4.7833893, 3.35083381], [-7.31355564, -4.91838953, 6.34983341], [-9.84455578, -5.79238817, 1.03683368], [-7.53355543, -6.13238833, -1.91016682], [-5.25055544, -3.12938807, -2.61016663], [-4.4865555, -2.53838846, -6.2571667], [-2.01055555, -0.45138857, -8.16116663], [-0.94755556, 6.0746105, -10.76916655], [-2.91355555, 5.4476116, -7.59116658], [-3.44855563, 2.69861105, -5.02816661], [-7.06655543, 1.58561209, -5.04616674], [-8.55455534, -0.103389, -2.03016671], [-1.13155556, 9.12460974, 2.60583297], [-3.54555552, 11.34461096, 0.683833

In [8]:
class NAAnoEnc:
    """Run this everytime we need a new set of embeddings"""
    def __init__(self, verbose=False):
        self.nAAno_emb = {}   # basically just make it easier to embed down the line
        self.verbose = verbose

    def initialize(self):
        self.nAAno_emb = {aa_id: get_embedding(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEnc initialized")
        return True

    def get_aa_id(self, emb_vector):
        aa_id = None
        for code, key in self.nAAno_emb.items():
            if key == emb_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Embedding vector {emb_vector} presented does not exist")

        return aa_id


def get_embedding(aa_id: str):
    """
    use this in these two scenarios:
    \n    - generating embeddings after updating feature vector
    \n    - inference
    :param aa_id: single letter amino acid reference code
    :returns: feature vector representing a given (valid) amino acid
    """
    # sanity check
    if aa_id not in AA_IDS:
        raise ValueError(f"{aa_id} not in valid AA ID list")

    # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
    embedding = [
        MOLECULAR_WEIGHTS[aa_id],
        NET_CHARGES[aa_id],
        ISOELECTRIC_PTS[aa_id],
        HYDROPHOBICITY_IDXS[aa_id],
        HALF_LIFE[aa_id],
    ]
    embedding += FUNCTIONAL_FP[aa_id]
    return embedding


def encoder_check():
    encoder = NAAnoEnc(verbose=True)
    encoder.initialize()
    for aa_code, aa_vect in encoder.nAAno_emb.items():
        print(f"{aa_code} -- {aa_vect}")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = get_embedding(aa)
        if aa_str == encoder.get_aa_id(aa_emb):
            print(f"{aa_str}: str <-> vect aligned")

encoder_check()  # note: all good

NAAnoEnc initialized
A -- [0.09, 0, 6.02, 41, 4.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C -- [0.12, 0, 5.02, 49, 1.2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]
D -- [0.13, -1, 2.98, -55, 1.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
E -- [0.15, -1, 3.22, -31, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
F -- [0.17, 0, 5.48, 100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
G -- [0.08, 0, 5.97, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H -- [0.16, 0, 7.59, 8, 3.5, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
I -- [0.13, 0, 5.98, 100, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
K -- [0.15, 1, 9.47, -23, 1.3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
L -- [0.13, 0, 5.98, 97, 5.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
M -- [0.15, 0, 5.75, 74, 30, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
N -- [0.13, 0, 5.41, -28, 1.4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
P -- [0.12, 0, 6.3, -46, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Q -- [0.15, 0, 5.65, -10, 0.8, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
R -- [0.17, 1,